<a href="https://colab.research.google.com/github/Parag003/Ml_lab/blob/main/LAB_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#generate the dataset


In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
X = np.random.randn(100, 1)
X = np.hstack([X + i*np.random.randn(100, 1) for i in range(1, 8)])
y = X.sum(axis=1) + np.random.randn(100)

df = pd.DataFrame(X, columns=[f"Feature_{i}" for i in range(1, 8)])
df['Target'] = y
print(df.head())


   Feature_1  Feature_2  Feature_3  Feature_4  Feature_5  Feature_6  \
0  -0.918657   1.212289  -1.990271  -5.880996   5.127602   5.038646   
1  -0.558910   0.983305  -1.818807  -2.535764   9.408819  -5.671256   
2   0.304974   2.813791   2.889569   0.668663  -6.345149   5.865324   
3   0.720753   3.630634   3.354141   1.710952   4.337876   9.656857   
4  -0.395439  -2.989492  -0.296858  -2.034415  -3.487366   2.246456   

   Feature_7     Target  
0  -3.162347   0.364549  
1   7.204800   6.496142  
2  -4.282717   2.010576  
3  -8.336199  14.612738  
4 -11.130558 -18.522169  


#Question 2 ) RIDGE

In [2]:
from sklearn.metrics import r2_score

def ridge_gradient_descent(X, y, lr, reg_param, num_iter):
    n, m = X.shape
    weights = np.zeros(m)
    bias = 0

    for _ in range(num_iter):
        y_pred = X.dot(weights) + bias
        error = y_pred - y
        weights -= lr * (X.T.dot(error) + reg_param * weights) / n
        bias -= lr * error.mean()

    return weights, bias

learning_rates = [0.0001, 0.001, 0.01]
reg_params = [1e-5, 0.001, 1, 10]

best_r2, best_params = -1, None
for lr in learning_rates:
    for reg in reg_params:
        weights, bias = ridge_gradient_descent(df.iloc[:, :-1].values, df['Target'].values, lr, reg, 1000)
        y_pred = df.iloc[:, :-1].values.dot(weights) + bias
        r2 = r2_score(df['Target'], y_pred)
        if r2 > best_r2:
            best_r2, best_params = r2, (lr, reg)

print("Best Params:", best_params, "Best R2:", best_r2)


Best Params: (0.01, 1e-05) Best R2: 0.9949727818773032


#Question 3 BOSTON HOUSE

In [4]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

california = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(california.data, california.target, test_size=0.2, random_state=42)

ridge_cv = RidgeCV(alphas=[0.1, 1.0, 10.0]).fit(X_train, y_train)
lasso_cv = LassoCV(alphas=[0.1, 1.0, 10.0]).fit(X_train, y_train)

y_pred_ridge = ridge_cv.predict(X_test)
y_pred_lasso = lasso_cv.predict(X_test)

print("RidgeCV R2:", r2_score(y_test, y_pred_ridge))
print("LassoCV R2:", r2_score(y_test, y_pred_lasso))


RidgeCV R2: 0.5764371556839779
LassoCV R2: 0.5318167610318159
